In [2]:
from google.colab import userdata

GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")

%cd /content
!git clone https://{GITHUB_TOKEN}@github.com/Bassey-data/Auto-insurance-claim-frequency.git

%cd /content/Auto-insurance-claim-frequency
!git config user.email "basseysamuel404@gmail.com"
!git config user.name "Bassey-data"

print("Setup complete")

/content
Cloning into 'Auto-insurance-claim-frequency'...
remote: Enumerating objects: 13, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 13 (delta 1), reused 10 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (13/13), 5.36 KiB | 1.79 MiB/s, done.
Resolving deltas: 100% (1/1), done.
/content/Auto-insurance-claim-frequency
Setup complete


In [3]:
pip install statsmodels liac-arff lightgbm pyarrow shap

  Preparing metadata (setup.py) ... done
  Created wheel for liac-arff: filename=liac_arff-2.5.0-py3-none-any.whl size=11717 sha256=0f1dc3e022d47f6ec3935a051739c5ced866bc06d6e09428f4886b792622c165
  Stored in directory: /root/.cache/pip/wheels/a9/ac/cf/c2919807a5c623926d217c0a18eb5b457e5c19d242c3b5963a
Successfully built liac-arff


Import all useful modules

In [ ]:

import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy.io import arff
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_poisson_deviance
import lightgbm as lgb
import shap
import matplotlib.pyplot as plt
import seaborn as sns
import os
import json
import warnings
warnings.filterwarnings("ignore")


In [ ]:
# Plot styling
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 110

In [ ]:
def parse_arff(path):
    columns = []
    data_start = None

    with open(path, "r", encoding="utf-8", errors="replace") as f:
        lines = f.readlines()

    for i, line in enumerate(lines):
        stripped = line.strip()
        if stripped.lower().startswith("@attribute"):
            parts = stripped.split(maxsplit=2)
            columns.append(parts[1])
        elif stripped.lower().startswith("@data"):
            data_start = i + 1
            break

    data_lines = [l.strip() for l in lines[data_start:] if l.strip()]

    rows = []
    for line in data_lines:
        values = [v.strip().strip("'").strip('"') for v in line.split(",")]
        rows.append(values)

    return pd.DataFrame(rows, columns=columns)


df = parse_arff("/content/drive/MyDrive/ACQsci.arff")

print(df.head())

In [ ]:
df.dtypes

fix numeric and categorical columns data types

In [ ]:
# used an f string for readable output
print(f"Shape: {df.shape}")
print(f"\nMissing values:\n{df.isna().sum()}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")

In [ ]:
df_converted = df.apply(pd.to_numeric, errors="ignore")

for col in df_converted.select_dtypes(include="object").columns:
    df_converted[col] = df_converted[col].astype("category")

df = df_converted
print(df.dtypes)

In [ ]:
print(f"Shape: {df.shape}")
print(f"\nMissing values:\n{df.isna().sum()}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")

In [ ]:
os.makedirs("data/processed", exist_ok=True)

df.to_parquet("data/processed/freMTPL2freq.parquet", index=False)

print("Saved successfully")
print(f"Parquet size: {os.path.getsize('data/processed/freMTPL2freq.parquet') / 1e6:.1f} MB")

In [ ]:
print(os.path.exists("/content/Auto-insurance-claim-frequency/data/processed/freMTPL2freq.parquet"))


In [ ]:
%cd /content/Auto-insurance-claim-frequency
!git add .
!git commit -m "Add data preparation notebook and processed parquet file"
!git push

In [ ]:
%cd /content/Auto-insurance-claim-frequency
!ls -la
!ls -la data/processed/
!ls -la notebooks/ sce

In [ ]:
%cd /content/Auto-insurance-claim-frequency

# Create notebooks folder and copy notebook into it
!mkdir -p notebooks
!cp "/content/drive/MyDrive/Colab Notebooks/Insurance_claim.ipynb" notebooks/01_data_prep.ipynb

# The parquet is gitignored which is correct - data files shouldn't be on GitHub
# Let's confirm notebooks got copied
!ls notebooks/

In [ ]:
!git add notebooks/
!git commit -m "Add data preparation notebook"
!git push

In [ ]:
df = pd.read_parquet("/content/Auto-insurance-claim-frequency/data/processed/freMTPL2freq.parquet")
print(df.shape)
print(df.head())

Check total clims and total exposure and calculate portfolio frequency

In [ ]:
total_claims = df["ClaimNb"].sum()
total_exposure = df["Exposure"].sum()
portfolio_frequency = total_claims / total_exposure

print(f"Total claims: {total_claims:,}")
print(f"Total exposure (policy-years): {total_exposure:,.1f}")
print(f"Portfolio frequency: {portfolio_frequency:.4f} claims per policy-year")

Check claim distribution

In [ ]:
claim_counts = df["ClaimNb"].value_counts().sort_index()
print(claim_counts)
print()
print(f"Policies with zero claims: {(df['ClaimNb'] == 0).mean() * 100:.2f}%")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

ax.bar(claim_counts.index.astype(str), claim_counts.values, color="steelblue")
ax.set_yscale("log")
ax.set_title("Distribution of ClaimNb (log scale)")
ax.set_xlabel("Number of claims")
ax.set_ylabel("Number of policies (log scale)")

plt.tight_layout()
plt.show()